# Investigating the influence of heat waves on work-related accident rates in Louisiana

## Importing Necessities

In [1]:
import warnings
warnings.filterwarnings("ignore")
from pathlib import Path

import numpy as np
import pandas as pd

import statsmodels.api as sm
import statsmodels.formula.api as smf

from sklearn.metrics import mean_absolute_error, mean_squared_error

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

## Loading the datasets

In [2]:
PROJECT_ROOT = Path.cwd().parent
DATA_FINAL = PROJECT_ROOT / "data" / "final"
model_file = DATA_FINAL / "parish_day_modeling_dataset.csv"

In [3]:
model_df = pd.read_csv(model_file)

print("Shape:", model_df.shape)
print("\nColumns:")
print(model_df.columns.tolist())

model_df.head()

Shape: (240739, 31)

Columns:
['parish', 'lcd_station_id', 'lcd_station_name', 'station_latitude', 'station_longitude', 'date', 'avg_temp', 'avg_dew_point', 'avg_relative_humidity', 'avg_wind_speed', 'max_temp', 'min_temp', 'avg_temp_c', 'min_temp_c', 'avg_dew_point_c', 'avg_wind_speed_ms', 'vapor_pressure_hpa', 'avg_apparent_temp_c', 'min_apparent_temp_c', 'apparent_temp_85th_percentile', 'exceeds_threshold', 'run_id', 'run_length', 'heatwave_flag', 'accident_count', 'fips', 'population', 'median_household_income', 'rucc_code', 'urban_rural_status', 'heatwave_binary']


,parish,lcd_station_id,lcd_station_name,station_latitude,station_longitude,date,avg_temp,avg_dew_point,avg_relative_humidity,avg_wind_speed,max_temp,min_temp,avg_temp_c,min_temp_c,avg_dew_point_c,avg_wind_speed_ms,vapor_pressure_hpa,avg_apparent_temp_c,min_apparent_temp_c,apparent_temp_85th_percentile,exceeds_threshold,run_id,run_length,heatwave_flag,accident_count,fips,population,median_household_income,rucc_code,urban_rural_status,heatwave_binary
0,Acadia,WBAN:00381,"JENNINGS AIRPORT, LA US",30.243,-92.673,2015-01-01,6.770833,4.251389,84.444444,3.800000,8.6,5.2,6.770833,5.2,4.251389,3.800000,8.269414,2.839740,1.268907,28.893006,False,1,168,non-heat-wave,0,22001,57576,44412,2.0,Urban,0
1,Acadia,WBAN:00381,"JENNINGS AIRPORT, LA US",30.243,-92.673,2015-01-02,13.173611,13.100000,99.472222,2.743056,18.5,8.8,13.173611,8.8,13.100000,2.743056,15.046863,12.218937,7.845326,28.893006,False,1,168,non-heat-wave,0,22001,57576,44412,2.0,Urban,0
2,Acadia,WBAN:00381,"JENNINGS AIRPORT, LA US",30.243,-92.673,2015-01-03,16.840278,16.733333,99.305556,3.327778,20.5,12.0,16.840278,12.0,16.733333,3.327778,19.008862,16.783758,11.943480,28.893006,False,1,168,non-heat-wave,0,22001,57576,44412,2.0,Urban,0
3,Acadia,WBAN:00381,"JENNINGS AIRPORT, LA US",30.243,-92.673,2015-01-04,8.555556,5.820833,83.666667,4.259722,11.4,4.0,8.555556,4.0,5.820833,4.259722,9.224931,4.617977,0.062422,28.893006,False,1,168,non-heat-wave,0,22001,57576,44412,2.0,Urban,0
4,Acadia,WBAN:00381,"JENNINGS AIRPORT, LA US",30.243,-92.673,2015-01-05,3.455556,-0.893056,74.527778,3.583333,9.0,0.0,3.455556,0.0,-0.893056,3.583333,5.720057,-1.165159,-4.620714,28.893006,False,1,168,non-heat-wave,0,22001,57576,44412,2.0,Urban,0


## Preprocessing

In [4]:
# Parsing date
model_df["date"] = pd.to_datetime(model_df["date"])

# Time features
model_df["year"] = model_df["date"].dt.year
model_df["month"] = model_df["date"].dt.month

# Making sure that accident count exists and is numeric
model_df["accident_count"] = pd.to_numeric(model_df["accident_count"], errors="coerce").fillna(0)

# Standardizing heatwave indicator
if "heatwave_binary" in model_df.columns:
    model_df["heatwave_binary"] = pd.to_numeric(model_df["heatwave_binary"], errors="coerce").fillna(0).astype(int)
elif "heatwave_flag" in model_df.columns:
    model_df["heatwave_binary"] = pd.to_numeric(model_df["heatwave_flag"], errors="coerce").fillna(0).astype(int)
elif "heatwave" in model_df.columns:
    # handling string labels if present
    model_df["heatwave_binary"] = (
        model_df["heatwave"]
        .astype(str)
        .str.lower()
        .isin(["1", "true", "yes", "heat-wave", "heatwave"])
        .astype(int)
    )
else:
    raise ValueError("No heatwave indicator column found.")

print(model_df[["date", "year", "month", "accident_count", "heatwave_binary"]].head())

        date  year  month  accident_count  heatwave_binary
0 2015-01-01  2015      1               0                0
1 2015-01-02  2015      1               0                0
2 2015-01-03  2015      1               0                0
3 2015-01-04  2015      1               0                0
4 2015-01-05  2015      1               0                0


### Performing Missingness Check

In [5]:
important_cols = [
    "accident_count",
    "heatwave_binary",
    "avg_apparent_temp_c",
    "min_apparent_temp_c",
    "avg_relative_humidity",
    "population",
    "median_household_income",
    "urban_rural_status"
]

missing_summary = model_df[important_cols].isna().sum().sort_values(ascending=False)
missing_summary

avg_relative_humidity      8951
accident_count                0
avg_apparent_temp_c           0
heatwave_binary               0
min_apparent_temp_c           0
population                    0
median_household_income       0
urban_rural_status            0
dtype: int64

#### We have some avg_relative_humidity missing data

### Removing implausible apparent temperatures

In [6]:
clean_df = model_df.copy()

temp_mask = (
    clean_df["avg_apparent_temp_c"].between(-20, 60, inclusive="both") &
    clean_df["min_apparent_temp_c"].between(-30, 50, inclusive="both")
)

before_rows = len(clean_df)
clean_df = clean_df[temp_mask].copy()
after_rows = len(clean_df)

print(f"Removed {before_rows - after_rows} rows with implausible apparent temperature values.")

Removed 25 rows with implausible apparent temperature values.


### Performing Humidity Imputation

In [7]:
clean_df["avg_relative_humidity"] = pd.to_numeric(clean_df["avg_relative_humidity"], errors="coerce")
humidity_median = clean_df["avg_relative_humidity"].median()
clean_df["avg_relative_humidity"] = clean_df["avg_relative_humidity"].fillna(humidity_median)

### Standardizing urban/rural field

In [8]:
clean_df["urban_rural_status"] = clean_df["urban_rural_status"].astype(str).str.strip()

### Inspecting heat-wave concentration by month
#### We are doing this to justify the hot-season sensitivity model

In [9]:
month_heatwave_summary = (
    clean_df.groupby("month")
    .agg(
        n_days=("heatwave_binary", "size"),
        heatwave_days=("heatwave_binary", "sum"),
        heatwave_rate=("heatwave_binary", "mean"),
        mean_accidents=("accident_count", "mean")
    )
    .reset_index()
)

month_heatwave_summary

,month,n_days,heatwave_days,heatwave_rate,mean_accidents
0,1,21276,0,0.000000,0.008601
1,2,19482,0,0.000000,0.010215
2,3,21403,0,0.000000,0.009298
3,4,20671,0,0.000000,0.009288
4,5,21287,61,0.002866,0.009020
5,6,20358,2198,0.107967,0.010905
6,7,20972,6575,0.313513,0.009823
7,8,19288,4974,0.257881,0.009747
8,9,18652,777,0.041658,0.010508
9,10,19310,6,0.000311,0.010150


#### We can observe that June, July and August have significant heatwave days, and therefore can consider these as hot months

## Performing train, validation and test split

In [10]:
train_df = clean_df[clean_df["year"].between(2015, 2023)].copy()
valid_df = clean_df[clean_df["year"] == 2024].copy()
test_df  = clean_df[clean_df["year"] == 2025].copy()

print("Train shape:", train_df.shape)
print("Validation shape:", valid_df.shape)
print("Test shape:", test_df.shape)

print("\nPositive accident-day rate")
print("Train:", (train_df["accident_count"] > 0).mean())
print("Valid:", (valid_df["accident_count"] > 0).mean())
print("Test :", (test_df["accident_count"] > 0).mean())

Train shape: (204425, 33)
Validation shape: (23101, 33)
Test shape: (13188, 33)

Positive accident-day rate
Train: 0.009592760180995474
Valid: 0.007315700619020821
Test : 0.007961783439490446


In [11]:
for d in [train_df, valid_df, test_df]:
    d["log_population"] = np.log(d["population"])

## Poisson regression for the Full Year 

### Defining the Formula

In [12]:
poisson_formula = """
accident_count ~ 
    avg_apparent_temp_c + 
    avg_relative_humidity + 
    heatwave_binary + 
    median_household_income + 
    C(urban_rural_status) + 
    C(month)
"""
print(poisson_formula)


accident_count ~ 
    avg_apparent_temp_c + 
    avg_relative_humidity + 
    heatwave_binary + 
    median_household_income + 
    C(urban_rural_status) + 
    C(month)



### Fitting the Full Year Poisson Model

In [13]:
poisson_model_full = smf.glm(
    formula=poisson_formula,
    data=train_df,
    family=sm.families.Poisson(),
    offset=train_df["log_population"]
).fit()

print(poisson_model_full.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:         accident_count   No. Observations:               204425
Model:                            GLM   Df Residuals:                   204408
Model Family:                 Poisson   Df Model:                           16
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -10326.
Date:                Thu, 19 Mar 2026   Deviance:                       16715.
Time:                        21:24:27   Pearson chi2:                 2.60e+05
No. Iterations:                     8   Pseudo R-squ. (CS):          0.0001084
Covariance Type:            nonrobust                                         
                                     coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------------
Intercept   

### Evaluation

In [14]:
def poisson_mean_deviance(y_true, y_pred, eps=1e-12):
    """
    Mean Poisson deviance for count predictions.
    """
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.clip(np.asarray(y_pred, dtype=float), eps, None)

    term = np.where(
        y_true == 0,
        y_pred,
        y_true * np.log(y_true / y_pred) - (y_true - y_pred)
    )
    return 2 * np.mean(term)

def evaluate_count_model(model, df_eval, train_reference_df, label="Evaluation"):
    y_true = df_eval["accident_count"].values
    y_pred = model.predict(df_eval, offset=df_eval["log_population"])

    # Null model: training mean accident rate per person * evaluation population
    train_rate = train_reference_df["accident_count"].sum() / train_reference_df["population"].sum()
    null_pred = train_rate * df_eval["population"].values

    results = {
        "dataset": label,
        "n": len(df_eval),
        "actual_mean": y_true.mean(),
        "predicted_mean": y_pred.mean(),
        "mae": mean_absolute_error(y_true, y_pred),
        "rmse": np.sqrt(mean_squared_error(y_true, y_pred)),
        "poisson_deviance_model": poisson_mean_deviance(y_true, y_pred),
        "poisson_deviance_null": poisson_mean_deviance(y_true, null_pred),
    }
    results["deviance_improvement"] = (
        results["poisson_deviance_null"] - results["poisson_deviance_model"]
    )
    return pd.Series(results)

full_valid_results = evaluate_count_model(
    poisson_model_full, valid_df, train_df, "Validation (Full Year, Offset)"
)

full_test_results = evaluate_count_model(
    poisson_model_full, test_df, train_df, "Test (Full Year, Offset)"
)

pd.DataFrame([full_valid_results, full_test_results])

,dataset,n,actual_mean,predicted_mean,mae,rmse,poisson_deviance_model,poisson_deviance_null,deviance_improvement
0,"Validation (Full Year, Offset)",23101,0.007359,0.009883,0.016851,0.085651,0.065819,0.066092,0.000274
1,"Test (Full Year, Offset)",13188,0.008113,0.010000,0.017751,0.091248,0.075438,0.075366,-0.000072


### Incidence Rate Ratios (IRRs)

In [15]:
coef_table = pd.DataFrame({
    "coef": poisson_model_full.params,
    "std_err": poisson_model_full.bse,
    "p_value": poisson_model_full.pvalues
})

coef_table["IRR"] = np.exp(coef_table["coef"])
coef_table["IRR_lower_95"] = np.exp(coef_table["coef"] - 1.96 * coef_table["std_err"])
coef_table["IRR_upper_95"] = np.exp(coef_table["coef"] + 1.96 * coef_table["std_err"])

coef_table.sort_values("p_value").head(20)

,coef,std_err,p_value,IRR,IRR_lower_95,IRR_upper_95
Intercept,-15.558145,0.198561,0.000000,1.750586e-07,1.186220e-07,2.583461e-07
avg_relative_humidity,-0.005243,0.002180,0.016193,9.947711e-01,9.905290e-01,9.990312e-01
C(month)[T.12],-0.139271,0.116843,0.233280,8.699921e-01,6.919212e-01,1.093891e+00
C(month)[T.2],0.124841,0.111409,0.262475,1.132968e+00,9.107188e-01,1.409454e+00
C(month)[T.11],-0.114465,0.118663,0.334733,8.918428e-01,7.067733e-01,1.125373e+00
heatwave_binary,0.087252,0.102301,0.393716,1.091172e+00,8.929209e-01,1.333439e+00
C(month)[T.10],0.099129,0.123990,0.424005,1.104209e+00,8.659818e-01,1.407971e+00
avg_apparent_temp_c,0.003645,0.004644,0.432498,1.003652e+00,9.945582e-01,1.012828e+00
C(month)[T.3],0.055924,0.115089,0.627026,1.057517e+00,8.439598e-01,1.325113e+00
C(month)[T.6],0.063754,0.147092,0.664701,1.065831e+00,7.988785e-01,1.421987e+00


In [16]:
heatwave_row = coef_table.loc[coef_table.index == "heatwave_binary"].copy()

print("Heatwave effect:")
display(heatwave_row)

Heatwave effect:


,coef,std_err,p_value,IRR,IRR_lower_95,IRR_upper_95
heatwave_binary,0.087252,0.102301,0.393716,1.091172,0.892921,1.333439


## Poisson regression for the Hot-season subset (June–August)

In [17]:
hot_months = [6, 7, 8]

train_hot = train_df[train_df["month"].isin(hot_months)].copy()
valid_hot = valid_df[valid_df["month"].isin(hot_months)].copy()
test_hot  = test_df[test_df["month"].isin(hot_months)].copy()

print("Train hot-season shape:", train_hot.shape)
print("Validation hot-season shape:", valid_hot.shape)
print("Test hot-season shape:", test_hot.shape)

print("\nHeatwave rate in hot-season data")
print("Train:", train_hot["heatwave_binary"].mean())
print("Valid:", valid_hot["heatwave_binary"].mean())
print("Test :", test_hot["heatwave_binary"].mean())

Train hot-season shape: (51085, 34)
Validation hot-season shape: (5856, 34)
Test hot-season shape: (3677, 34)

Heatwave rate in hot-season data
Train: 0.2146422628951747
Valid: 0.3235997267759563
Test : 0.24122926298613


### Fitting the Hot-Season Poisson Model

In [26]:
poisson_model_hot = smf.glm(
    formula=poisson_formula,
    data=train_hot,
    family=sm.families.Poisson(),
    offset=train_hot["log_population"]
).fit()

print(poisson_model_hot.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:         accident_count   No. Observations:                51085
Model:                            GLM   Df Residuals:                    51077
Model Family:                 Poisson   Df Model:                            7
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -2645.3
Date:                Thu, 19 Mar 2026   Deviance:                       4268.2
Time:                        21:29:44   Pearson chi2:                 6.73e+04
No. Iterations:                     8   Pseudo R-squ. (CS):          0.0001867
Covariance Type:            nonrobust                                         
                                     coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------------
Intercept   

### Evaluation of the Hot-Season Poisson Model

In [27]:
hot_valid_results = evaluate_count_model(
    poisson_model_hot, valid_hot, train_hot, "Validation (Hot Season, Offset)"
)

hot_test_results = evaluate_count_model(
    poisson_model_hot, test_hot, train_hot, "Test (Hot Season, Offset)"
)

pd.DataFrame([hot_valid_results, hot_test_results])

,dataset,n,actual_mean,predicted_mean,mae,rmse,poisson_deviance_model,poisson_deviance_null,deviance_improvement
0,"Validation (Hot Season, Offset)",5856,0.010075,0.010485,0.020043,0.101055,0.085371,0.085031,-0.000340
1,"Test (Hot Season, Offset)",3677,0.011150,0.010389,0.021105,0.107440,0.098511,0.098155,-0.000356


## Comparing Full Year vs Hot-Season Results

In [28]:
results_table = pd.DataFrame([
    full_valid_results,
    full_test_results,
    hot_valid_results,
    hot_test_results
])

results_table

,dataset,n,actual_mean,predicted_mean,mae,rmse,poisson_deviance_model,poisson_deviance_null,deviance_improvement
0,"Validation (Full Year, Offset)",23101,0.007359,0.009883,0.016851,0.085651,0.065819,0.066092,0.000274
1,"Test (Full Year, Offset)",13188,0.008113,0.010000,0.017751,0.091248,0.075438,0.075366,-0.000072
2,"Validation (Hot Season, Offset)",5856,0.010075,0.010485,0.020043,0.101055,0.085371,0.085031,-0.000340
3,"Test (Hot Season, Offset)",3677,0.011150,0.010389,0.021105,0.107440,0.098511,0.098155,-0.000356


## Comparing Heat-wave Coefficients Across Models

In [29]:
def extract_heatwave_effect(glm_result, model_name):
    params = glm_result.params
    bse = glm_result.bse
    pvals = glm_result.pvalues

    if "heatwave_binary" not in params.index:
        return pd.Series({
            "model": model_name,
            "coef": np.nan,
            "IRR": np.nan,
            "p_value": np.nan,
            "IRR_lower_95": np.nan,
            "IRR_upper_95": np.nan
        })

    coef = params["heatwave_binary"]
    se = bse["heatwave_binary"]

    return pd.Series({
        "model": model_name,
        "coef": coef,
        "IRR": np.exp(coef),
        "p_value": pvals["heatwave_binary"],
        "IRR_lower_95": np.exp(coef - 1.96 * se),
        "IRR_upper_95": np.exp(coef + 1.96 * se)
    })

heatwave_compare = pd.DataFrame([
    extract_heatwave_effect(poisson_model_full, "Full Year"),
    extract_heatwave_effect(poisson_model_hot, "Hot Season (Jun-Aug)")
])

heatwave_compare

,model,coef,IRR,p_value,IRR_lower_95,IRR_upper_95
0,Full Year,0.087252,1.091172,0.393716,0.892921,1.333439
1,Hot Season (Jun-Aug),0.003022,1.003026,0.980942,0.782774,1.285251


## Overdispersion Check

In [31]:
train_mean = train_df["accident_count"].mean()
train_var = train_df["accident_count"].var()

print("Training mean accident count :", train_mean)
print("Training variance accident count:", train_var)

Training mean accident count : 0.009710162651339121
Training variance accident count: 0.009850728520805244
